In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch

print(transformers.__version__)


/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LLM_MODEL = "microsoft/deberta-v3-small"

In [3]:

# Data loading + preprocessing
csv_path = "data/train.csv"
df = pd.read_csv(csv_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["label"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    )
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = (
        "Prompt:\n" + df["prompt"].astype(str) +
        "\n\nResponse A:\n" + df["response_a"].astype(str) +
        "\n\nResponse B:\n" + df["response_b"].astype(str)
    )
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["label"].value_counts())

Loaded 57477 rows
label
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [4]:


# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "label"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]


def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 5748/5748 [00:00<00:00, 6426.30 examples/s]


In [5]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 98553.10it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

In [6]:


training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=5e-7,
    per_device_train_batch_size=1,
    num_train_epochs=1,
    fp16=False,
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="none",
    # log every 1000 steps:
    save_strategy="epoch",
    logging_strategy="steps",
    max_grad_norm=1.0,
    logging_steps=50,
    # evaluation_strategy="steps",
    eval_steps=100,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.228866
100,1.744436
150,1.609815
200,1.888696
250,2.110892
300,1.484850
350,1.324111
400,1.344890
450,1.322592
500,1.685306


In [ ]:
# Save the trained model and tokenizer
trainer.save_model("./model")
tokenizer.save_pretrained("./model")


In [ ]:

# Evaluate and show metrics
# metrics = trainer.evaluate(eval_dataset=val_dataset)
# print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])

In [ ]:
# 1) Load test data
test_df = pd.read_csv("data/test.csv")

# 2) Prepare text in same format as training
test_df["text"] = (
    "Prompt:\n" + test_df["prompt"].astype(str) +
    "\n\nResponse A:\n" + test_df["response_a"].astype(str) +
    "\n\nResponse B:\n" + test_df["response_b"].astype(str)
)

test_dataset = Dataset.from_pandas(test_df[["text"]])

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



In [ ]:
# a) single record test check
demo = test_df.iloc[[0]]
tok = tokenizer(
    demo["text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt",
)
print("input_ids shape", tok["input_ids"].shape)
print("attention_mask sum", tok["attention_mask"].sum().item())

# # b) model forward
model.eval()
with torch.no_grad():
    out = model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
print("logits:", out.logits)
print("nan?", torch.isnan(out.logits).any().item(), "inf?", torch.isinf(out.logits).any().item())

In [ ]:
model_cpu = model.to("cpu")
tok_cpu = {k: v.to("cpu") for k,v in tok.items()}
with torch.no_grad():
    out = model_cpu(**tok_cpu)
print(out.logits[:2])

In [ ]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-small", num_labels=3
).eval()

demo_tok = tokenizer(
    ["Prompt:\nhello\n\nResponse A:\nA\n\nResponse B:\nB"],
    truncation=True, padding="max_length", max_length=256, return_tensors="pt"
)

with torch.no_grad():
    baseline_out = baseline_model(**demo_tok)
print("baseline logits:", baseline_out.logits)
print("baseline nan?", torch.isnan(baseline_out.logits).any().item())

In [ ]:
trained = AutoModelForSequenceClassification.from_pretrained("./model").eval()
with torch.no_grad():
    test_out = trained(**demo_tok)
print("trained logits:", test_out.logits)
print("trained nan?", torch.isnan(test_out.logits).any().item())

In [ ]:


# 4) Predict
preds = trainer.predict(test_dataset)
print("Raw predictions shape:", preds)
logits = preds.predictions  # shape (N, 3)
print("Logits shape:", logits)
logits_tensor = torch.from_numpy(logits).float()

if torch.isnan(logits_tensor).any() or torch.isinf(logits_tensor).any():
    raise ValueError("Logits contain NaN or Inf; check model/prediction pipeline")

# stable softmax to avoid numeric instabilities
logits_stable = logits_tensor - logits_tensor.max(dim=-1, keepdim=True).values
exp_scores = torch.exp(logits_stable)
probs_tensor = exp_scores / exp_scores.sum(dim=-1, keepdim=True)
probs = probs_tensor.cpu().numpy()

# 5) save to submission file
out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv("data/submission_pred.csv", index=False)
print(out.head())


In [ ]:
print(train_dataset.column_names)

In [ ]:
print(test_dataset.column_names)
print(test_dataset[0])

In [ ]:
probs